In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

# ==========================================
# 1. MOTOR MATEMÁTICO GENÉRICO
# ==========================================
def avaliar_grid(params, var1_name, var1_array, var2_name, var2_array):
    V1, V2 = np.meshgrid(var1_array, var2_array, indexing='ij')

    p = params.copy()
    p[var1_name] = V1
    p[var2_name] = V2

    g1 = p['P'] * (p['R'] - p['LR'] - p['Cs'] - p['G']) + (1 - p['P']) * (-p['Cs'])
    g2 = (1 - p['P']) * (p['I'] - p['Ca']) + p['P'] * (p['I'] - p['L'] + p['G'] - p['Ca'])

    n_IA = (p['R'] >= 0) & ((p['I'] - p['L']) >= g2)
    n_IF = (g1 >= 0) & (g2 >= (p['I'] - p['L']))
    n_WA = (0 >= p['R']) & (p['I'] >= (p['I'] - p['alpha'] * p['Ca']))
    n_WF = (0 >= g1) & ((p['I'] - p['alpha'] * p['Ca']) >= p['I'])

    IA = n_IA
    IF = n_IF & ~IA
    WA = n_WA & ~IA & ~IF
    WF = n_WF & ~IA & ~IF & ~WA

    return IA, IF, WA, WF

def aplicar_variacao(val_base, var_array, tipo_var):
    if tipo_var == 'Percentual (%)':
        return val_base * (1 + var_array / 100.0)
    return var_array

def achar_cruzamento(x, y1, y2):
    diff = np.array(y1) - np.array(y2)
    cruzamentos = []
    for i in range(len(diff) - 1):
        if diff[i] * diff[i+1] < 0:
            x_int = x[i] - diff[i] * (x[i+1] - x[i]) / (diff[i+1] - diff[i])
            y_int = y1[i] + (y1[i+1] - y1[i]) * (x_int - x[i]) / (x[i+1] - x[i])
            cruzamentos.append((x_int, y_int))
    return cruzamentos

cores = {'IA': '#4CAF50', 'IF': '#F44336', 'WA': '#9E9E9E', 'WF': '#607D8B', 'NoEq': '#FFC107'}
labels_eq = {'IA': 'Implement / Accept', 'IF': 'Implement / Fight', 'WA': 'Withdraw / Accept', 'WF': 'Withdraw / Fight', 'NoEq': 'Absence of Pure Equilibrium'}

nomes_bonitos = {
    'R': "Consortium Expected Return 'R'",
    'LR': "Delay Cost 'LR' (in Millions)",
    'Cs': "Judicial Cost 'Cs' (in Millions)",
    'I': "Stakeholders Base Revenue 'I'",
    'L': "Estimated Local Economic Impact 'L'",
    'alpha': "Mobilization Cost Coefficient 'alpha'",
    'Ca': "Judicial Cost 'Ca' (in Millions)",
    'G': "Financial Compensation Offered 'G'",
    'P': "Probability of Offshore Institutional Success 'P'"
}

# ==========================================
# 2. INTERFACE DE USUÁRIO (DASHBOARD)
# ==========================================
style = {'description_width': '180px'}
layout = {'width': '450px'}
variaveis = ['R', 'LR', 'Cs', 'I', 'L', 'alpha', 'Ca', 'G', 'P']

lbl_cat1 = widgets.HTML("<h3 style='margin-bottom:0;'>1. Analysis Selection</h3><hr style='margin:0 0 10px 0;'>")
w_tipo_grafico = widgets.Dropdown(options=['Area Evolution (2D Grid vs X Variable)', '2D Heatmap (Y Variable vs X Variable)', 'Expected Surf Payoff (1D)'], value='Area Evolution (2D Grid vs X Variable)', description='Chart Type:', style=style, layout=layout)
w_impacto = widgets.Dropdown(options=[('Low Impact (L=4.4)', 4.4), ('Medium Impact (L=8.8)', 8.8), ('High Impact (L=19.8)', 19.8)], value=4.4, description='Impact Scenario (L):', style=style, layout=layout)
box_cat1 = widgets.VBox([lbl_cat1, w_tipo_grafico, w_impacto], layout={'margin': '0 0 20px 0'})

lbl_cat2 = widgets.HTML("<h3 style='margin-bottom:0;'>2. Main Axis (X Variable)</h3><hr style='margin:0 0 10px 0;'>")
w_var_x = widgets.Dropdown(options=variaveis, value='Ca', description='Variable:', style=style, layout=layout)
w_tipo_var_x = widgets.Dropdown(options=['Absoluto', 'Percentual (%)'], value='Absoluto', description='Variation Type:', style=style, layout=layout)
w_min_x = widgets.FloatText(value=0.0, description='Minimum Value:', style=style, layout=layout)
w_max_x = widgets.FloatText(value=22.0, description='Maximum Value:', style=style, layout=layout)
w_steps_x = widgets.IntText(value=21, description='Line Points:', style=style, layout=layout)
box_cat2 = widgets.VBox([lbl_cat2, w_var_x, w_tipo_var_x, w_min_x, w_max_x, w_steps_x], layout={'margin': '0 0 20px 0', 'border': '1px solid #ddd', 'padding': '10px'})

lbl_cat3 = widgets.HTML("<h3 style='margin-bottom:0;'>3. Background Matrix (Y and Z Grids)</h3><hr style='margin:0 0 10px 0;'>")
w_var_y = widgets.Dropdown(options=variaveis, value='P', description='Y Axis / Grid 1:', style=style, layout=layout)
w_min_y = widgets.FloatText(value=0.0, description='Min Y/Grid 1:', style=style, layout=layout)
w_max_y = widgets.FloatText(value=1.0, description='Max Y/Grid 1:', style=style, layout=layout)
w_var_z = widgets.Dropdown(options=variaveis, value='G', description='Grid 2 (Areas):', style=style, layout=layout)
w_min_z = widgets.FloatText(value=0.0, description='Min Grid 2:', style=style, layout=layout)
w_max_z = widgets.FloatText(value=6.6, description='Max Grid 2:', style=style, layout=layout)
w_res = widgets.IntText(value=100, description='Grid Resolution:', style=style, layout=layout)
box_cat3 = widgets.VBox([lbl_cat3, w_var_y, w_min_y, w_max_y, widgets.HTML("<br>"), w_var_z, w_min_z, w_max_z, widgets.HTML("<br>"), w_res], layout={'margin': '0 0 20px 0', 'border': '1px solid #ddd', 'padding': '10px'})

aba_config = widgets.VBox([box_cat1, box_cat2, box_cat3])

w_R = widgets.FloatText(value=100.0, description='Return (R):', style=style, layout=layout)
w_LR = widgets.FloatText(value=20.0, description='Delay Cost (LR):', style=style, layout=layout)
w_Cs = widgets.FloatText(value=2.0, description='Offshore Cost (Cs):', style=style, layout=layout)
w_I = widgets.FloatText(value=22.0, description='Surf Revenue (I):', style=style, layout=layout)
w_alpha = widgets.FloatText(value=0.1, description='Mobilization (alpha):', style=style, layout=layout)
w_Ca = widgets.FloatText(value=3.0, description='Stakeholder Cost (Ca):', style=style, layout=layout)
w_G = widgets.FloatText(value=4.4, description='Compensation (G):', style=style, layout=layout)
w_P = widgets.FloatText(value=0.5, description='Institutional Success (P):', style=style, layout=layout)

aba_params = widgets.VBox([widgets.HTML("<h3>Baseline Model Parameters</h3><hr>"), w_R, w_LR, w_Cs, w_I, w_Ca, w_alpha, w_G, w_P], layout={'padding': '10px'})

abas = widgets.Tab(children=[aba_config, aba_params])
abas.set_title(0, 'Chart Controls')
abas.set_title(1, 'Baseline Parameters')

# ==========================================
# 3. ATUALIZAÇÃO AUTOMÁTICA
# ==========================================
def obter_limites_padrao(var_name, L_val, I_val, tipo_var):
    if tipo_var == 'Percentual (%)': return -50.0, 50.0
    if var_name in ['P', 'alpha']: return 0.0, 1.0
    if var_name == 'G': return 0.0, 1.5 * L_val
    if var_name == 'Ca': return 0.0, I_val
    if var_name in ['LR', 'Cs']: return 0.0, 100.0
    if var_name == 'L': return 0.0, 19.8
    if var_name in ['R', 'I']: return 0.0, 200.0
    return 0.0, 100.0

def atualizar_limites(*args):
    L_val, I_val = w_impacto.value, w_I.value
    w_min_x.value, w_max_x.value = obter_limites_padrao(w_var_x.value, L_val, I_val, w_tipo_var_x.value)
    w_min_y.value, w_max_y.value = obter_limites_padrao(w_var_y.value, L_val, I_val, 'Absoluto')
    w_min_z.value, w_max_z.value = obter_limites_padrao(w_var_z.value, L_val, I_val, 'Absoluto')

for w in [w_var_x, w_var_y, w_var_z, w_tipo_var_x, w_impacto, w_I]: w.observe(atualizar_limites, 'value')
atualizar_limites()

# ==========================================
# 4. BOTÕES E LÓGICA DE PLOTAGEM
# ==========================================
btn_plot = widgets.Button(description='Generate Chart', button_style='primary', icon='play')
btn_export = widgets.Button(description='Export PNG', button_style='success', icon='download')
w_insight = widgets.HTML(value="", layout={'margin': '15px 0 0 0', 'padding': '10px', 'border-left': '4px solid #4CAF50'})
output = widgets.Output()
figura_atual = None

def plotar_grafico(b):
    global figura_atual
    with output:
        clear_output(wait=True)
        w_insight.value = ""

        params_base = {'R': w_R.value, 'LR': w_LR.value, 'Cs': w_Cs.value, 'I': w_I.value, 'L': w_impacto.value, 'alpha': w_alpha.value, 'Ca': w_Ca.value, 'G': w_G.value, 'P': w_P.value}

        figura_atual, ax = plt.subplots(figsize=(8, 6))
        x_array_linhas = np.linspace(w_min_x.value, w_max_x.value, w_steps_x.value)
        x_array_hm = np.linspace(w_min_x.value, w_max_x.value, w_res.value)

        nome_x = nomes_bonitos.get(w_var_x.value, w_var_x.value)
        sufixo = '(%)' if w_tipo_var_x.value == 'Percentual (%)' else ''
        prefixo = 'Absolute Value of ' if w_tipo_var_x.value == 'Absoluto' else 'Variation of '

        if w_tipo_grafico.value == 'Area Evolution (2D Grid vs X Variable)':
            grid1_array = np.linspace(w_min_y.value, w_max_y.value, w_res.value)
            grid2_array = np.linspace(w_min_z.value, w_max_z.value, w_res.value)

            areas = {'IA': [], 'IF': [], 'WA': [], 'WF': [], 'NoEq': []}
            x_vals = aplicar_variacao(params_base[w_var_x.value], x_array_linhas, w_tipo_var_x.value)

            for x_val in x_vals:
                p_iter = params_base.copy()
                p_iter[w_var_x.value] = x_val
                IA, IF, WA, WF = avaliar_grid(p_iter, w_var_y.value, grid1_array, w_var_z.value, grid2_array)
                NoEq = ~IA & ~IF & ~WA & ~WF
                total = w_res.value ** 2
                areas['IA'].append(np.sum(IA) / total * 100)
                areas['IF'].append(np.sum(IF) / total * 100)
                areas['WA'].append(np.sum(WA) / total * 100)
                areas['WF'].append(np.sum(WF) / total * 100)
                areas['NoEq'].append(np.sum(NoEq) / total * 100)

            for key in ['IA', 'IF', 'WA', 'WF', 'NoEq']:
                if max(areas[key]) > 0:
                    ax.plot(x_array_linhas, areas[key], color=cores[key], marker='o', markersize=5, linewidth=2.5, label=labels_eq[key])

            # Insight limpo, usando tags semutais HTML em vez de cores inline rigorosas
            cruzamentos = achar_cruzamento(x_array_linhas, areas['IA'], areas['IF'])
            if cruzamentos:
                texto_cruzamento = "".join([f"<li>When <b>{w_var_x.value} = {c_x:.2f}</b>, the areas are equal at <b>{c_y:.1f}%</b>.</li>" for c_x, c_y in cruzamentos])
                w_insight.value = f"<div style='font-size:14px;'><b>Strategic Transition Point:</b> Where agreement and litigation occupy the same share of the uncertainty grid.<ul>{texto_cruzamento}</ul></div>"

            ax.set_xlabel(f"{prefixo}{nome_x} {sufixo}")
            ax.set_ylabel(f"Area in the {w_var_y.value} vs {w_var_z.value} scenario grid (%)")
            ax.set_ylim(-5, 105)
            ax.set_xticks(np.linspace(w_min_x.value, w_max_x.value, 11))
            ax.legend(title="Nash Equilibrium", loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=True, edgecolor='black')

        elif w_tipo_grafico.value == '2D Heatmap (Y Variable vs X Variable)':
            y_array = np.linspace(w_min_y.value, w_max_y.value, w_res.value)
            IA, IF, WA, WF = avaliar_grid(params_base, w_var_y.value, y_array, w_var_x.value, x_array_hm)
            NoEq = ~IA & ~IF & ~WA & ~WF

            Z = np.zeros((w_res.value, w_res.value))
            Z[IA], Z[IF], Z[WA], Z[WF], Z[NoEq] = 0, 1, 2, 3, 4
            cmap = mcolors.ListedColormap([cores['IA'], cores['IF'], cores['WA'], cores['WF'], cores['NoEq']])
            norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

            X_mesh, Y_mesh = np.meshgrid(x_array_hm, y_array)
            ax.pcolormesh(X_mesh, Y_mesh, Z, cmap=cmap, norm=norm, shading='auto')

            w_insight.value = "<div style='font-size:14px;'><b>Heatmap Reading:</b> Heatmaps do not have single line-crossing points. Color boundaries identify the decision-transition zones.</div>"

            ax.set_xlabel(nomes_bonitos.get(w_var_x.value, w_var_x.value))
            ax.set_ylabel(nomes_bonitos.get(w_var_y.value, w_var_y.value))
            chaves = ['IA', 'IF', 'WA', 'WF', 'NoEq']
            legend_elements = [Patch(facecolor=cores[chaves[int(v)]], edgecolor='black', label=labels_eq[chaves[int(v)]]) for v in np.unique(Z)]
            ax.legend(handles=legend_elements, title="Nash Equilibrium", loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=True, edgecolor='black')

        else:
            x_vals = aplicar_variacao(params_base[w_var_x.value], x_array_linhas, w_tipo_var_x.value)
            payoff_acc, payoff_fig = [], []
            for x_val in x_vals:
                p_iter = params_base.copy()
                p_iter[w_var_x.value] = x_val
                acc = p_iter['I'] - p_iter['L']
                fig = (1 - p_iter['P']) * (p_iter['I'] - p_iter['Ca']) + p_iter['P'] * (p_iter['I'] - p_iter['L'] + p_iter['G'] - p_iter['Ca'])
                payoff_acc.append(acc)
                payoff_fig.append(fig)

            ax.plot(x_array_linhas, payoff_acc, 'k--', linewidth=2, label='Agreement')
            ax.plot(x_array_linhas, payoff_fig, 'gray', linewidth=2, label='Fight')

            cruzamentos = achar_cruzamento(x_array_linhas, payoff_acc, payoff_fig)
            if cruzamentos:
                texto_cruzamento = "".join([f"<li><b>{w_var_x.value} = {c_x:.2f}</b> (Resulting in an expected payoff of <b>{c_y:.1f}</b>).</li>" for c_x, c_y in cruzamentos])
                w_insight.value = f"<div style='font-size:14px;'><b>Indifference Point (Price of Peace):</b> Where the expected value of accepting and fighting is identical:<ul>{texto_cruzamento}</ul></div>"

            ax.set_xlabel(f"{prefixo}{nome_x} {sufixo}")
            ax.set_ylabel('Expected Payoff of the Surf Community')
            ax.set_xticks(np.linspace(w_min_x.value, w_max_x.value, 11))
            ax.legend(title="Rational Decision", loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=True, edgecolor='black')

        ax.grid(True, linestyle='--', alpha=0.6)
        figura_atual.tight_layout()
        plt.show()

def exportar_grafico(b):
    global figura_atual
    with output:
        if figura_atual is not None:
            filename = "interactive_model_figure.png"
            figura_atual.savefig(filename, dpi=300, bbox_inches='tight')
            files.download(filename)
        else:
            print("Generate a chart first.")

btn_plot.on_click(plotar_grafico)
btn_export.on_click(exportar_grafico)

display(abas, widgets.HTML("<br>"), widgets.HBox([btn_plot, btn_export]), w_insight, output)

## Deterministic article-figure workflow

This cell mirrors `run_peniche_offshore_analysis.py` and regenerates the article figures and traceability CSV files with the manuscript calibration.


In [ ]:
# Deterministic article-figure workflow aligned with run_peniche_offshore_analysis.py
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

try:
    PROJECT_ROOT = Path(__file__).resolve().parents[2]
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

OUTPUT_DIR = PROJECT_ROOT
FIGURE_DIR = OUTPUT_DIR / "article_figures"
TRACE_DIR = OUTPUT_DIR / "traceability"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TRACE_DIR.mkdir(parents=True, exist_ok=True)

COLOR_MAP = {
    "Implement / Accept": "#4CAF50",
    "Implement / Fight": "#F44336",
    "Withdraw / Accept": "#9E9E9E",
    "Withdraw / Fight": "#607D8B",
    "Absence of Pure Equilibrium": "#FFC107",
}
EQ_LEVELS = list(COLOR_MAP)

PARAMS = {"R": 100.0, "LR": 20.0, "Cs": 2.0, "I": 22.0, "L": 4.4, "alpha": 0.1, "Ca": 3.0, "G": 4.4, "P": 0.5}
IMPACT_SCENARIOS = {"low_impact": 0.2 * PARAMS["I"], "medium_impact": 0.4 * PARAMS["I"], "high_impact": 0.9 * PARAMS["I"]}


def equilibrium_class(p):
    g1 = p["P"] * (p["R"] - p["LR"] - p["Cs"] - p["G"]) + (1 - p["P"]) * (-p["Cs"])
    g2 = (1 - p["P"]) * (p["I"] - p["Ca"]) + p["P"] * (p["I"] - p["L"] + p["G"] - p["Ca"])
    n_ia = (p["R"] >= 0) & ((p["I"] - p["L"]) >= g2)
    n_if = (g1 >= 0) & (g2 >= (p["I"] - p["L"]))
    n_wa = (0 >= p["R"]) & (p["I"] >= (p["I"] - p["alpha"] * p["Ca"]))
    n_wf = (0 >= g1) & ((p["I"] - p["alpha"] * p["Ca"]) >= p["I"])
    result = np.full(np.size(g1), "Absence of Pure Equilibrium", dtype=object)
    result[np.asarray(n_wf).reshape(-1)] = "Withdraw / Fight"
    result[(np.asarray(n_wa) & ~np.asarray(n_wf)).reshape(-1)] = "Withdraw / Accept"
    result[(np.asarray(n_if) & ~np.asarray(n_wa) & ~np.asarray(n_wf)).reshape(-1)] = "Implement / Fight"
    result[np.asarray(n_ia).reshape(-1)] = "Implement / Accept"
    return result.reshape(np.shape(g1))


def grid_equilibria(params, x_name, x_values, y_name, y_values):
    x_mesh, y_mesh = np.meshgrid(x_values, y_values, indexing="ij")
    p = params.copy()
    p[x_name] = x_mesh
    p[y_name] = y_mesh
    z = equilibrium_class(p)
    rows = pd.DataFrame({x_name: x_mesh.ravel(), y_name: y_mesh.ravel(), "equilibrium": z.ravel()})
    return rows, z


def area_evolution(params, varying_name, varying_values, x_values=None, y_values=None):
    if x_values is None:
        x_values = np.linspace(0, 1, 100)
    if y_values is None:
        y_values = np.linspace(0, 1.5 * params["L"], 100)
    rows = []
    for value in varying_values:
        p = params.copy()
        p[varying_name] = value
        _, z = grid_equilibria(p, "P", x_values, "G", y_values)
        total = z.size
        for eq in EQ_LEVELS:
            rows.append({"variable": varying_name, "value": value, "equilibrium": eq, "area_percent": (z == eq).sum() / total * 100})
    return pd.DataFrame(rows)


def plot_heatmap(rows, z, x_name, y_name, title, xlab, ylab, filename):
    x_values = np.sort(rows[x_name].unique())
    y_values = np.sort(rows[y_name].unique())
    z_int = np.vectorize(lambda eq: EQ_LEVELS.index(eq))(z)
    cmap = mcolors.ListedColormap([COLOR_MAP[eq] for eq in EQ_LEVELS])
    norm = mcolors.BoundaryNorm(np.arange(len(EQ_LEVELS) + 1) - 0.5, cmap.N)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.pcolormesh(x_values, y_values, z_int.T, cmap=cmap, norm=norm, shading="auto")
    ax.set_title(title, fontfamily="serif", fontweight="bold")
    ax.set_xlabel(xlab, fontfamily="serif")
    ax.set_ylabel(ylab, fontfamily="serif")
    ax.grid(True, linestyle="--", alpha=0.35)
    handles = [Patch(facecolor=COLOR_MAP[eq], edgecolor="black", label=eq) for eq in EQ_LEVELS if eq in set(rows["equilibrium"])]
    ax.legend(handles=handles, title="Nash Equilibrium", loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=300)
    plt.close(fig)


def plot_area(rows, title, xlab, filename):
    fig, ax = plt.subplots(figsize=(10, 6))
    for eq in EQ_LEVELS:
        eq_rows = rows[rows["equilibrium"] == eq]
        if eq_rows["area_percent"].max() > 0:
            ax.plot(eq_rows["value"], eq_rows["area_percent"], color=COLOR_MAP[eq], marker="o", linewidth=2.5, markersize=5, label=eq)
    ax.set_title(title, fontfamily="serif", fontweight="bold")
    ax.set_xlabel(xlab, fontfamily="serif")
    ax.set_ylabel("Area in the P vs G scenario grid (%)", fontfamily="serif")
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(title="Nash Equilibrium", loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=300)
    plt.close(fig)


def copy_article_figure(source_name, article_name):
    (FIGURE_DIR / article_name).write_bytes((FIGURE_DIR / source_name).read_bytes())


params_tbl = pd.DataFrame({"symbol": list(PARAMS), "value": list(PARAMS.values())})
params_tbl.to_csv(OUTPUT_DIR / "model_parameters.csv", index=False)

g1 = PARAMS["P"] * (PARAMS["R"] - PARAMS["LR"] - PARAMS["Cs"] - PARAMS["G"]) + (1 - PARAMS["P"]) * (-PARAMS["Cs"])
g2 = (1 - PARAMS["P"]) * (PARAMS["I"] - PARAMS["Ca"]) + PARAMS["P"] * (PARAMS["I"] - PARAMS["L"] + PARAMS["G"] - PARAMS["Ca"])
baseline = pd.DataFrame({
    "offshore_action": ["Implement", "Implement", "Withdraw", "Withdraw"],
    "surf_action": ["Accept", "Fight", "Accept", "Fight"],
    "offshore_payoff": [PARAMS["R"], g1, 0, 0],
    "surf_payoff": [PARAMS["I"] - PARAMS["L"], g2, PARAMS["I"], PARAMS["I"] - PARAMS["alpha"] * PARAMS["Ca"]],
})
baseline["total_payoff"] = baseline["offshore_payoff"] + baseline["surf_payoff"]
baseline["better_for_offshore"] = [baseline.offshore_payoff[0] >= baseline.offshore_payoff[2], baseline.offshore_payoff[1] >= baseline.offshore_payoff[3], baseline.offshore_payoff[2] >= baseline.offshore_payoff[0], baseline.offshore_payoff[3] >= baseline.offshore_payoff[1]]
baseline["better_for_surf"] = [baseline.surf_payoff[0] >= baseline.surf_payoff[1], baseline.surf_payoff[1] >= baseline.surf_payoff[0], baseline.surf_payoff[2] >= baseline.surf_payoff[3], baseline.surf_payoff[3] >= baseline.surf_payoff[2]]
baseline["is_nash"] = baseline["better_for_offshore"] & baseline["better_for_surf"]
baseline["is_pareto_max"] = baseline["total_payoff"] == baseline["total_payoff"].max()
baseline.to_csv(OUTPUT_DIR / "baseline_payoff_matrix.csv", index=False)

p_values = np.linspace(0, 1, 100)
low = PARAMS.copy(); low["L"] = IMPACT_SCENARIOS["low_impact"]
g_values_low = np.linspace(0, 1.5 * low["L"], 100)
pxg_rows, pxg_z = grid_equilibria(low, "G", g_values_low, "P", p_values)
pxg_rows.to_csv(TRACE_DIR / "figure_01_probability_compensation_low_impact.csv", index=False)
plot_heatmap(pxg_rows, pxg_z, "G", "P", "Sensitivity Analysis: Probability (P) vs. Compensation (G)", "Financial Compensation Offered (G)", "Probability of Offshore Institutional Success (P)", "probability_compensation_low_impact.png")
copy_article_figure("probability_compensation_low_impact.png", "grafico1.png")

ca_variations = np.arange(-50, 55, 5)
for slug, value in IMPACT_SCENARIOS.items():
    p = PARAMS.copy(); p["L"] = value
    area = area_evolution(p, "Ca", PARAMS["Ca"] * (1 + ca_variations / 100), y_values=np.linspace(0, 1.5 * p["L"], 100))
    area["variation_percent"] = np.repeat(ca_variations, len(EQ_LEVELS))
    area.to_csv(TRACE_DIR / f"stakeholder_cost_area_{slug}.csv", index=False)
    plot_area(area.assign(value=area["variation_percent"]), f"Evolution of Equilibrium Areas by Stakeholder Cost - {slug.replace('_', ' ').title()}", "Variation of Stakeholder Cost Ca (%)", f"stakeholder_cost_area_{slug}.png")
copy_article_figure("stakeholder_cost_area_low_impact.png", "grafico2.png")
copy_article_figure("stakeholder_cost_area_medium_impact.png", "grafico3.png")
copy_article_figure("stakeholder_cost_area_high_impact.png", "grafico4.png")

l_area = area_evolution(PARAMS, "L", np.linspace(0, 0.9 * PARAMS["I"], 21), y_values=np.linspace(0, 1.5 * 0.9 * PARAMS["I"], 100))
l_area.to_csv(TRACE_DIR / "expected_local_loss_area.csv", index=False)
plot_area(l_area, "Evolution of Equilibrium Areas by Expected Local Loss", "Expected Local Loss L", "expected_local_loss_area.png")
copy_article_figure("expected_local_loss_area.png", "grafico5.png")

gl_rows, gl_z = grid_equilibria(PARAMS, "G", np.linspace(0, 1.5 * PARAMS["I"], 100), "L", np.linspace(0, 0.9 * PARAMS["I"], 100))
gl_rows.to_csv(TRACE_DIR / "compensation_loss_heatmap.csv", index=False)
plot_heatmap(gl_rows, gl_z, "G", "L", "Strategic Frontier: Compensation (G) vs. Local Loss (L)", "Financial Compensation Offered (G)", "Expected Local Loss (L)", "compensation_loss_heatmap.png")
copy_article_figure("compensation_loss_heatmap.png", "grafico6.png")

lr_area = area_evolution(low, "LR", np.linspace(0, PARAMS["R"], 21), y_values=np.linspace(0, 1.5 * low["L"], 100))
lr_area.to_csv(TRACE_DIR / "delay_cost_area_low_impact.csv", index=False)
plot_area(lr_area, "Evolution of Equilibrium Areas by Delay Cost - Low Impact", "Delay Cost LR", "delay_cost_area_low_impact.png")
copy_article_figure("delay_cost_area_low_impact.png", "grafico7.png")

for slug in ["low_impact", "high_impact"]:
    p = PARAMS.copy(); p["L"] = IMPACT_SCENARIOS[slug]; p["G"] = p["L"]
    cost_rows, cost_z = grid_equilibria(p, "Ca", np.linspace(0, PARAMS["I"], 100), "Cs", np.linspace(0, PARAMS["R"], 100))
    cost_rows.to_csv(TRACE_DIR / f"stakeholder_consortium_cost_heatmap_{slug}.csv", index=False)
    plot_heatmap(cost_rows, cost_z, "Ca", "Cs", f"Legal-Cost Asymmetry: {slug.replace('_', ' ').title()}", "Stakeholder Cost (Ca)", "Offshore Consortium Cost (Cs)", f"stakeholder_consortium_cost_heatmap_{slug}.png")
copy_article_figure("stakeholder_consortium_cost_heatmap_low_impact.png", "grafico8.png")
copy_article_figure("stakeholder_consortium_cost_heatmap_high_impact.png", "grafico9.png")

figure_index = pd.DataFrame({
    "manuscript_file": [f"images/grafico{i}.png" for i in range(1, 10)],
    "artifact_file": [f"article_figures/grafico{i}.png" for i in range(1, 10)],
    "source_traceability": [
        "traceability/figure_01_probability_compensation_low_impact.csv",
        "traceability/stakeholder_cost_area_low_impact.csv",
        "traceability/stakeholder_cost_area_medium_impact.csv",
        "traceability/stakeholder_cost_area_high_impact.csv",
        "traceability/expected_local_loss_area.csv",
        "traceability/compensation_loss_heatmap.csv",
        "traceability/delay_cost_area_low_impact.csv",
        "traceability/stakeholder_consortium_cost_heatmap_low_impact.csv",
        "traceability/stakeholder_consortium_cost_heatmap_high_impact.csv",
    ],
    "description": [
        "P x G equilibrium heatmap, low-impact scenario.",
        "P x G x Ca area evolution, low-impact scenario.",
        "P x G x Ca area evolution, medium-impact scenario.",
        "P x G x Ca area evolution, high-impact scenario.",
        "P x G x L area evolution.",
        "G x L equilibrium heatmap.",
        "P x G x LR area evolution, low-impact scenario.",
        "Ca x Cs equilibrium heatmap, low-impact scenario.",
        "Ca x Cs equilibrium heatmap, high-impact scenario.",
    ],
})
figure_index.to_csv(OUTPUT_DIR / "article_figure_index.csv", index=False)
print(f"Deterministic notebook outputs written to {OUTPUT_DIR}")
